### Create Dataframe

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# -----------------------------
# Sample customer data
# -----------------------------
rows_customers = [
    (1, "Asha",  "IN", True),
    (2, "Bob",   "US", False),
    (3, "Chen",  "CN", True),
    (4, "Diana", "US", None),
    (None, "Ghost", "UK", False),  # NULL key for join demonstration
]

# -----------------------------
# Sample orders data
# -----------------------------
rows_orders = [
    (101, 1,    120.0, "IN"),
    (102, 1,     80.0, "IN"),
    (103, 2,     50.0, "US"),
    (104, 5,     30.0, "DE"),   # No matching customer_id
    (105, 3,    200.0, "CN"),
    (106, None,  15.0, "UK"),   # NULL key will not match
    (107, 3,     40.0, "CN"),
    (108, 2,     75.0, "US"),
]

# -----------------------------
# Define customer schema
# -----------------------------
schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("vip",         T.BooleanType(), True),
])

# -----------------------------
# Define orders schema
# -----------------------------
schema_orders = T.StructType([
    T.StructField("order_id",    T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount",      T.DoubleType(),  True),
    T.StructField("country",     T.StringType(),  True),  # Same column name for join demo
])

# -----------------------------
# Create DataFrames
# -----------------------------
df_customers = spark.createDataFrame(
    rows_customers,
    schema_customers
)

df_orders = spark.createDataFrame(
    rows_orders,
    schema_orders
)

# -----------------------------
# Display DataFrames
# -----------------------------
display(df_customers)
display(df_orders)

In [0]:
# Inner join
df_orders_customers = df_orders.join(df_customers, on="customer_id", how="inner")
display(df_orders_customers)

In [0]:
# Left join
df_orders_customers = df_orders.join(df_customers, on="customer_id", how="left")
display(df_orders_customers)

### Alias

In [0]:
# Alias
df_o_alias = df_orders.alias("o")
display(df_o_alias)
df_o_alias.show()



In [0]:
o, c = df_orders.alias("o"), df_customers.alias("c")
df_inner = o.join(c, o.customer_id == c.customer_id, how="inner")
display(df_inner)


In [0]:
df_inner_clean = (
    o.join(c, on="customer_id", how="inner")
    .select("order_id", "customer_id", "amount", 
            F.col("o.country").alias("order_country"),
                 "name",
            F.col("c.country").alias("customer_country"),
             "vip")
)
display(df_inner_clean)

In [0]:
o.join(c, on="customer_id", how="full").display()

In [0]:
o.join(c, on="customer_id", how="left_semi").display()

### Multi Column Join

In [0]:
o.join(c, on=["customer_id", "country"], how="left_semi").display()

### Use Broadcast Join

In [0]:
from pyspark.sql.functions import broadcast
o.join(broadcast(c), on="customer_id", how="left_semi").display()